# Explore pattern clusters
Run `python run.py all` first. This notebook loads the results and lets you browse groups and look-alikes.

In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath('..'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
import config as cfg

best = json.loads((cfg.RESULTS_DIR / 'best.json').read_text())
MODEL, METHOD = best['model'], best['method']   # change to compare others
metrics = pd.read_csv(cfg.RESULTS_DIR / 'metrics.csv')
metrics.sort_values(['model', 'method'])

In [ ]:
labels = pd.read_csv(cfg.RESULTS_DIR / MODEL / METHOD / 'labels.csv')
knn = pd.read_csv(cfg.RESULTS_DIR / MODEL / 'neighbors.csv')
xy = np.load(cfg.RESULTS_DIR / MODEL / 'xy.npy')
print(MODEL, METHOD, '| groups:', labels.cluster.max() + 1, '| unique seeds:', (labels.cluster < 0).sum())
labels[labels.cluster >= 0].cluster.value_counts().describe()

In [ ]:
def show(seeds, title='', cols=8):
    seeds = list(seeds)
    rows = (len(seeds) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(2.2 * cols, 2.2 * rows), squeeze=False)
    for ax in axes.flat: ax.axis('off')
    for ax, s in zip(axes.flat, seeds):
        ax.imshow(Image.open(cfg.PROCESSED_DIR / f'seed_{s:04d}.png')); ax.set_title(s, fontsize=9)
    fig.suptitle(title); plt.tight_layout(); plt.show()

def group_of(seed):
    c = labels.set_index('seed').cluster[seed]
    show(labels[labels.cluster == c].seed if c >= 0 else [seed], f'seed {seed}: group {c}')

def lookalikes(seed, k=7):
    nb = knn[knn.seed == seed].sort_values('rank')
    show([seed] + nb.neighbor.tolist()[:k], f'seed {seed} and nearest: ' + ', '.join(f'{s:.2f}' for s in nb.similarity[:k]))

In [ ]:
SEED = 1   # try a seed you care about
group_of(SEED)
lookalikes(SEED)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))
l = labels.cluster.to_numpy()
ax.scatter(*xy[l < 0].T, s=6, c='#bbb')
ax.scatter(*xy[l >= 0].T, s=10, c=l[l >= 0] % 20, cmap='tab20')
for s in [SEED]:
    i = labels.seed.tolist().index(s); ax.annotate(str(s), xy[i], color='red')
ax.set_title(f'{MODEL} / {METHOD}'); plt.show()